# Recorded OOD + overlay eval (all 10 checkpoints)

Scores `weights/{orig,clean}-seed{42–46}.keras` on `thinkpad`, `vivo`, `flow`, `thinkpad-2`, `flow-2`, then on the existing eval-only noise / RIR / DIR feature files. Does not remake overlays. Resume-friendly via `run_eval.py`.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd()
if HERE.name != "cnn-latest":
    HERE = Path("training/notebooks/cnn-latest").resolve()
OOD_CSV = HERE / "results" / "ood_seeds.csv"
OVERLAY_CSV = HERE / "results" / "overlay_seeds.csv"

for stage in ("ood", "overlay"):
    display(Markdown(f"## {stage}"))
    result = subprocess.run(
        ["uv", "run", "python", "run_eval.py", "--stage", stage],
        cwd=HERE,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"run_eval.py --stage {stage} exited {result.returncode}")

ood = pd.read_csv(OOD_CSV)
pair = {"thinkpad": "thinkpad-2", "flow": "flow-2"}
rows = []
for (variant, seed), g in ood.groupby(["variant", "seed"]):
    acc = g.set_index("dataset")["accuracy"]
    rows.append({
        "variant": variant, "seed": seed,
        "thinkpad": acc["thinkpad"], "thinkpad-2": acc["thinkpad-2"],
        "gap_thinkpad": acc["thinkpad"] - acc["thinkpad-2"],
        "flow": acc["flow"], "flow-2": acc["flow-2"],
        "gap_flow": acc["flow"] - acc["flow-2"],
        "vivo": acc["vivo"],
    })
tab = pd.DataFrame(rows).sort_values(["variant", "seed"])
display(Markdown("### Recorded accuracy (draft Table layout)"))
display(tab.round(3))

n_pos = int((tab.gap_thinkpad > 0).sum() + (tab.gap_flow > 0).sum())
kb_cost_tp = tab.gap_thinkpad
rec_cost_tp = 1.0 - tab.thinkpad
kb_cost_fl = tab.gap_flow
rec_cost_fl = 1.0 - tab.flow
n_kb = int((kb_cost_tp > rec_cost_tp).sum() + (kb_cost_fl > rec_cost_fl).sum())
display(Markdown(
    f"Keyboard gap > 0 on **{n_pos}/20** pairs. "
    f"Keyboard cost > recording-setting cost on **{n_kb}/20**."
))

ov = pd.read_csv(OVERLAY_CSV)
merged = ov.merge(
    ood.rename(columns={"accuracy": "recorded"})[["variant", "seed", "dataset", "recorded"]],
    on=["variant", "seed", "dataset"],
)
merged["tax"] = merged["recorded"] - merged["accuracy"]
display(Markdown("### Overlay tax (recorded − overlay), mean over seeds"))
display(
    merged.groupby(["variant", "dataset", "overlay"])[["accuracy", "tax"]]
    .mean().unstack("overlay").round(3)
)